# Mixture Model (Posterior Sampling) - Fine-Tuning

Mixture of Bayesian Walk PS and Bayesian Threshold PS, sharing the same posterior.

At each stage, both policies produce a role distribution per agent:
- **Walk PS**: stick with prob (1 - eps_switch), or switch via posterior marginal
- **Threshold PS**: switch if marginal gap > delta, otherwise stick

Final per-agent distribution = w * walk_dist + (1 - w) * thresh_dist

Params: tau_prior, epsilon, eps_switch, delta, w (5 params)

Caching: trajectories depend only on (tau_prior, epsilon), so we precompute those
and cheaply sweep (eps_switch, delta, w).

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import product
from scipy.optimize import minimize

from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

In [2]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

Loaded 66 team-rounds across 6 environments


## Model Definition

In [3]:
def posterior_marginal(prior, agent_i):
    """Marginalize the (3,3,3) joint posterior to get P(role) for agent_i."""
    marg = np.sum(prior, axis=tuple(j for j in range(3) if j != agent_i))
    total = marg.sum()
    return marg / total if total > 0 else np.ones(3) / 3.0


def walk_ps_dist(prior, agent_i, prev_role, epsilon_switch):
    """Walk PS policy: stick or switch via posterior marginal."""
    marg = posterior_marginal(prior, agent_i)
    if prev_role is None:
        return marg
    stick = np.zeros(3)
    stick[prev_role] = 1.0
    return (1.0 - epsilon_switch) * stick + epsilon_switch * marg


def thresh_ps_dist(prior, agent_i, prev_role, delta):
    """Threshold PS policy: switch if marginal gap > delta."""
    marg = posterior_marginal(prior, agent_i)
    if prev_role is None:
        return marg
    current_prob = marg[prev_role]
    candidates = [r for r in range(3) if r != prev_role and (marg[r] - current_prob) > delta]
    if not candidates:
        dist = np.zeros(3)
        dist[prev_role] = 1.0
        return dist
    candidate_probs = np.array([marg[r] for r in candidates])
    candidate_probs = candidate_probs / candidate_probs.sum()
    dist = np.zeros(3)
    for i, r in enumerate(candidates):
        dist[r] = candidate_probs[i]
    return dist


def make_mixture_ps(tau_prior=2.0, epsilon=0.2, epsilon_switch=0.3, delta=0.1, w=0.5):
    """Mixture of Walk PS and Threshold PS, sharing the same posterior."""
    def predict_fn(record):
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        results = []
        turn_idx = 0
        prev_roles = None

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            per_agent = []
            for i in range(3):
                prev_r = prev_roles[i] if prev_roles is not None else None
                d_walk = walk_ps_dist(prior, i, prev_r, epsilon_switch)
                d_thresh = thresh_ps_dist(prior, i, prev_r, delta)
                per_agent.append(w * d_walk + (1.0 - w) * d_thresh)

            predicted_dist = {}
            for r0 in range(3):
                for r1 in range(3):
                    for r2 in range(3):
                        combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                        predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

            results.append({
                'predicted_dist': predicted_dist,
                'human_combo': human_combo,
                'model_marginal': np.mean(per_agent, axis=0),
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = human_roles
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        return results
    return predict_fn


def precompute_trajectories(records, tau_prior, epsilon):
    """Precompute posterior + game state per stage for each record."""
    trajectories = []
    for record in records:
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        turn_idx = 0
        prev_roles = None
        stages = []

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            stages.append({
                'prior': prior.copy(),
                'human_combo': human_combo,
                'prev_roles': prev_roles,
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = list(human_roles)
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent_t = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent_t, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent_t, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent_t, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        trajectories.append(stages)
    return trajectories


def predict_from_trajectory(trajectory, epsilon_switch, delta, w):
    """Generate predictions from precomputed trajectory."""
    results = []
    for stage in trajectory:
        prior = stage['prior']
        prev_roles = stage['prev_roles']
        human_combo = stage['human_combo']

        per_agent = []
        for i in range(3):
            prev_r = prev_roles[i] if prev_roles is not None else None
            d_walk = walk_ps_dist(prior, i, prev_r, epsilon_switch)
            d_thresh = thresh_ps_dist(prior, i, prev_r, delta)
            per_agent.append(w * d_walk + (1.0 - w) * d_thresh)

        predicted_dist = {}
        for r0 in range(3):
            for r1 in range(3):
                for r2 in range(3):
                    combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                    predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

        results.append({
            'predicted_dist': predicted_dist,
            'human_combo': human_combo,
            'model_marginal': np.mean(per_agent, axis=0),
        })
    return results

## Fine-Tuning Helpers

In [4]:
metric = 'combo_r'

def pick_best(results, metric='combo_r'):
    valid = [r for r in results if not np.isnan(r.get(metric, float('nan')))]
    if not valid:
        return results[0]
    return max(valid, key=lambda r: r[metric])


def evaluate_mixture(records, tau_prior, epsilon, epsilon_switch, delta, w):
    """Run mixture model at given params (uncached, for scipy)."""
    results = run_predictions(records, make_mixture_ps(
        tau_prior=tau_prior, epsilon=epsilon,
        epsilon_switch=epsilon_switch, delta=delta, w=w))
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'epsilon': epsilon,
        'epsilon_switch': epsilon_switch, 'delta': delta, 'w': w,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_from_cache(records, trajectories, epsilon_switch, delta, w):
    """Evaluate using precomputed trajectories (fast)."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory(trajectories[idx], epsilon_switch, delta, w)
    results = run_predictions(records, predict_fn)
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'epsilon_switch': epsilon_switch, 'delta': delta, 'w': w,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_mixture(records, tau_prior_vals, eps_vals, eps_switch_vals, delta_vals, w_vals):
    """Cached grid search: precompute per (tau_prior, eps), sweep (eps_switch, delta, w)."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    outer_combos = list(product(tau_prior_vals, eps_vals))
    inner_combos = list(product(eps_switch_vals, delta_vals, w_vals))
    total = len(outer_combos) * len(inner_combos)
    count = 0

    for tp, eps in outer_combos:
        trajectories = precompute_trajectories(records, tp, eps)
        for eps_sw, delta, w in inner_combos:
            count += 1
            if count % 200 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            res = evaluate_from_cache(records, trajectories, eps_sw, delta, w)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results

## Aggregate Fine-Tuning

In [5]:
# Grid ranges
tau_prior_min, tau_prior_max, tau_prior_steps = 0.1, 20.0, 10
eps_min, eps_max, eps_steps = 0.001, 0.2, 10
eps_switch_vals = np.array([0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0])
delta_vals = np.array([0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5])
w_vals = np.array([0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0])

tau_prior_vals = np.linspace(tau_prior_min, tau_prior_max, tau_prior_steps)
eps_vals = np.geomspace(eps_min, eps_max, eps_steps)

total = len(tau_prior_vals) * len(eps_vals) * len(eps_switch_vals) * len(delta_vals) * len(w_vals)
print(f'Coarse grid: {len(tau_prior_vals)} x {len(eps_vals)} x {len(eps_switch_vals)} x {len(delta_vals)} x {len(w_vals)} = {total} points')
print(f'  tau_prior:   linspace({tau_prior_min}, {tau_prior_max}, {tau_prior_steps})')
print(f'  epsilon:     geomspace({eps_min}, {eps_max}, {eps_steps})')
print(f'  eps_switch:  {eps_switch_vals}')
print(f'  delta:       {delta_vals}')
print(f'  w:           {w_vals}')
print()

sweep_results = grid_search_mixture(records, tau_prior_vals, eps_vals,
                                     eps_switch_vals, delta_vals, w_vals)
best = pick_best(sweep_results, metric)
print(f'\nCoarse best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f} '
      f'eps_switch={best["epsilon_switch"]:.4f} delta={best["delta"]:.4f} w={best["w"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Coarse grid: 10 x 10 x 9 x 7 x 7 = 44100 points
  tau_prior:   linspace(0.1, 20.0, 10)
  epsilon:     geomspace(0.001, 0.2, 10)
  eps_switch:  [0.05 0.1  0.2  0.3  0.4  0.5  0.6  0.8  1.  ]
  delta:       [0.   0.01 0.05 0.1  0.2  0.3  0.5 ]
  w:           [0.  0.2 0.4 0.5 0.6 0.8 1. ]

  [200/44100] ...
  [400/44100] ...
  [600/44100] ...
  [800/44100] ...
  [1000/44100] ...
  [1200/44100] ...


/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:129: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(marg_m, marg_h)


  [1400/44100] ...
  [1600/44100] ...
  [1800/44100] ...
  [2000/44100] ...
  [2200/44100] ...
  [2400/44100] ...
  [2600/44100] ...
  [2800/44100] ...
  [3000/44100] ...
  [3200/44100] ...
  [3400/44100] ...
  [3600/44100] ...
  [3800/44100] ...
  [4000/44100] ...
  [4200/44100] ...
  [4400/44100] ...
  [4600/44100] ...
  [4800/44100] ...
  [5000/44100] ...
  [5200/44100] ...
  [5400/44100] ...
  [5600/44100] ...
  [5800/44100] ...
  [6000/44100] ...
  [6200/44100] ...
  [6400/44100] ...
  [6600/44100] ...
  [6800/44100] ...
  [7000/44100] ...
  [7200/44100] ...
  [7400/44100] ...
  [7600/44100] ...
  [7800/44100] ...
  [8000/44100] ...
  [8200/44100] ...
  [8400/44100] ...
  [8600/44100] ...
  [8800/44100] ...
  [9000/44100] ...
  [9200/44100] ...
  [9400/44100] ...
  [9600/44100] ...
  [9800/44100] ...
  [10000/44100] ...
  [10200/44100] ...
  [10400/44100] ...
  [10600/44100] ...
  [10800/44100] ...
  [11000/44100] ...
  [11200/44100] ...
  [11400/44100] ...
  [11600/44100] ...
  [

In [6]:
# Refine around coarse best
tp_step = (tau_prior_max - tau_prior_min) / tau_prior_steps
eps_ratio = (eps_max / eps_min) ** (1.0 / eps_steps)

fine_tp = np.linspace(
    max(tau_prior_min, best['tau_prior'] - tp_step),
    min(tau_prior_max, best['tau_prior'] + tp_step),
    8,
)
fine_eps = np.geomspace(
    max(eps_min, best['epsilon'] / eps_ratio),
    min(eps_max, best['epsilon'] * eps_ratio),
    8,
)

def refine_range(val, vals_arr, n=6):
    sorted_vals = np.sort(vals_arr)
    idx = np.searchsorted(sorted_vals, val)
    lo = sorted_vals[max(0, idx - 1)]
    hi = sorted_vals[min(len(sorted_vals) - 1, idx + 1)] if idx < len(sorted_vals) - 1 else sorted_vals[-1]
    return np.linspace(lo, hi, n)

fine_esw = refine_range(best['epsilon_switch'], eps_switch_vals)
fine_delta = refine_range(best['delta'], delta_vals)
fine_w = refine_range(best['w'], w_vals)

total_fine = len(fine_tp) * len(fine_eps) * len(fine_esw) * len(fine_delta) * len(fine_w)
print(f'Refined grid: {len(fine_tp)} x {len(fine_eps)} x {len(fine_esw)} x {len(fine_delta)} x {len(fine_w)} = {total_fine} points')
print(f'  tau_prior:  [{fine_tp[0]:.4f}, {fine_tp[-1]:.4f}]')
print(f'  epsilon:    [{fine_eps[0]:.6f}, {fine_eps[-1]:.6f}]')
print(f'  eps_switch: [{fine_esw[0]:.4f}, {fine_esw[-1]:.4f}]')
print(f'  delta:      [{fine_delta[0]:.4f}, {fine_delta[-1]:.4f}]')
print(f'  w:          [{fine_w[0]:.4f}, {fine_w[-1]:.4f}]')
print()

refined_results = grid_search_mixture(records, fine_tp, fine_eps,
                                       fine_esw, fine_delta, fine_w)
sweep_results.extend(refined_results)
best = pick_best(sweep_results, metric)
print(f'\nRefined best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f} '
      f'eps_switch={best["epsilon_switch"]:.4f} delta={best["delta"]:.4f} w={best["w"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Refined grid: 8 x 8 x 6 x 6 x 6 = 13824 points
  tau_prior:  [0.3211, 4.3011]
  epsilon:    [0.003443, 0.009934]
  eps_switch: [0.0500, 0.1000]
  delta:      [0.0000, 0.0100]
  w:          [0.5000, 0.8000]

  [200/13824] ...
  [400/13824] ...
  [600/13824] ...
  [800/13824] ...
  [1000/13824] ...
  [1200/13824] ...
  [1400/13824] ...
  [1600/13824] ...
  [1800/13824] ...
  [2000/13824] ...
  [2200/13824] ...
  [2400/13824] ...
  [2600/13824] ...
  [2800/13824] ...
  [3000/13824] ...
  [3200/13824] ...
  [3400/13824] ...
  [3600/13824] ...
  [3800/13824] ...
  [4000/13824] ...
  [4200/13824] ...
  [4400/13824] ...
  [4600/13824] ...
  [4800/13824] ...
  [5000/13824] ...
  [5200/13824] ...
  [5400/13824] ...
  [5600/13824] ...
  [5800/13824] ...
  [6000/13824] ...
  [6200/13824] ...
  [6400/13824] ...
  [6600/13824] ...
  [6800/13824] ...
  [7000/13824] ...
  [7200/13824] ...
  [7400/13824] ...
  [7600/13824] ...
  [7800/13824] ...
  [8000/13824] ...
  [8200/13824] ...
  [8400/13824] ...

In [7]:
# Scipy L-BFGS-B polishing
def objective(params):
    tp, eps, eps_sw, delta, w = params
    res = evaluate_mixture(records, tp, eps, eps_sw, delta, w)
    return -res[metric]

tp_lo = max(tau_prior_min, best['tau_prior'] - tp_step / 2)
tp_hi = min(tau_prior_max, best['tau_prior'] + tp_step / 2)
eps_lo = max(1e-6, best['epsilon'] / 2)
eps_hi = min(eps_max, best['epsilon'] * 2)
esw_lo = max(0.01, best['epsilon_switch'] - 0.1)
esw_hi = min(1.0, best['epsilon_switch'] + 0.1)
delta_lo = max(0.0, best['delta'] - 0.05)
delta_hi = min(1.0, best['delta'] + 0.05)
w_lo = max(0.0, best['w'] - 0.1)
w_hi = min(1.0, best['w'] + 0.1)

x0 = [best['tau_prior'], best['epsilon'], best['epsilon_switch'], best['delta'], best['w']]
bounds = [(tp_lo, tp_hi), (eps_lo, eps_hi), (esw_lo, esw_hi), (delta_lo, delta_hi), (w_lo, w_hi)]

print(f'Scipy optimization (metric={metric}) ...')
print(f'  bounds: tau_prior=[{tp_lo:.4f}, {tp_hi:.4f}], epsilon=[{eps_lo:.6f}, {eps_hi:.6f}]')
print(f'          eps_switch=[{esw_lo:.4f}, {esw_hi:.4f}], delta=[{delta_lo:.4f}, {delta_hi:.4f}], w=[{w_lo:.4f}, {w_hi:.4f}]')

opt_result = minimize(objective, x0, method='L-BFGS-B', bounds=bounds,
                      options={'maxiter': 50, 'ftol': 1e-6})
opt_tp, opt_eps, opt_esw, opt_delta, opt_w = opt_result.x
print(f'  Optimal: tau_prior={opt_tp:.4f} eps={opt_eps:.6f} eps_switch={opt_esw:.4f} '
      f'delta={opt_delta:.4f} w={opt_w:.4f}')
print(f'  objective={-opt_result.fun:.4f}')

opt_eval = evaluate_mixture(records, opt_tp, opt_eps, opt_esw, opt_delta, opt_w)
sweep_results.append(opt_eval)
best = pick_best(sweep_results, metric)
print(f'\nFinal best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f} '
      f'eps_switch={best["epsilon_switch"]:.4f} delta={best["delta"]:.4f} w={best["w"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Scipy optimization (metric=combo_r) ...
  bounds: tau_prior=[0.4633, 2.4533], epsilon=[0.002330, 0.009320]
          eps_switch=[0.0100, 0.1500], delta=[0.0000, 0.0500], w=[0.5800, 0.7800]
  Optimal: tau_prior=1.4583 eps=0.004660 eps_switch=0.0500 delta=0.0000 w=0.6800
  objective=0.4967

Final best: tau_prior=1.4583 eps=0.004660 eps_switch=0.0500 delta=0.0000 w=0.6800
  combo_r=0.5239  marg_r=0.5295  mean_ll=-9.3327


## Switch-Stage Fine-Tuning

In [8]:
def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)

        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0

        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])

        if max_stages == 0:
            continue

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        filtered[env_id] = dict(data)
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })

    return filtered


def evaluate_mixture_switch(records, tau_prior, epsilon, epsilon_switch, delta, w):
    """Run on full history, compute metrics only on switch stages."""
    full_results = run_predictions(records, make_mixture_ps(
        tau_prior=tau_prior, epsilon=epsilon,
        epsilon_switch=epsilon_switch, delta=delta, w=w))
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'tau_prior': tau_prior, 'epsilon': epsilon,
                'epsilon_switch': epsilon_switch, 'delta': delta, 'w': w,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'epsilon': epsilon,
        'epsilon_switch': epsilon_switch, 'delta': delta, 'w': w,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_switch_from_cache(records, trajectories, epsilon_switch, delta, w):
    """Evaluate switch-stage metrics using precomputed trajectories."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory(trajectories[idx], epsilon_switch, delta, w)
    full_results = run_predictions(records, predict_fn)
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'epsilon_switch': epsilon_switch, 'delta': delta, 'w': w,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'epsilon_switch': epsilon_switch, 'delta': delta, 'w': w,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_mixture_switch(records, tau_prior_vals, eps_vals, eps_switch_vals, delta_vals, w_vals):
    """Cached grid search optimizing switch-stage correlation."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    outer_combos = list(product(tau_prior_vals, eps_vals))
    inner_combos = list(product(eps_switch_vals, delta_vals, w_vals))
    total = len(outer_combos) * len(inner_combos)
    count = 0

    for tp, eps in outer_combos:
        trajectories = precompute_trajectories(records, tp, eps)
        for eps_sw, delta, w in inner_combos:
            count += 1
            if count % 200 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            res = evaluate_switch_from_cache(records, trajectories, eps_sw, delta, w)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results

In [9]:
# Coarse grid (same ranges)
print('=== Switch-Stage Fine-Tuning: Coarse Grid ===')
sw_total = len(tau_prior_vals) * len(eps_vals) * len(eps_switch_vals) * len(delta_vals) * len(w_vals)
print(f'{sw_total} points\n')

sw_sweep = grid_search_mixture_switch(records, tau_prior_vals, eps_vals,
                                       eps_switch_vals, delta_vals, w_vals)
best_sw = pick_best(sw_sweep, metric)
print(f'\nCoarse best: tau_prior={best_sw["tau_prior"]:.4f} eps={best_sw["epsilon"]:.6f} '
      f'eps_switch={best_sw["epsilon_switch"]:.4f} delta={best_sw["delta"]:.4f} w={best_sw["w"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

=== Switch-Stage Fine-Tuning: Coarse Grid ===
44100 points

  [200/44100] ...
  [400/44100] ...
  [600/44100] ...
  [800/44100] ...
  [1000/44100] ...
  [1200/44100] ...


/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:129: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(marg_m, marg_h)


  [1400/44100] ...
  [1600/44100] ...
  [1800/44100] ...
  [2000/44100] ...
  [2200/44100] ...
  [2400/44100] ...
  [2600/44100] ...
  [2800/44100] ...
  [3000/44100] ...
  [3200/44100] ...
  [3400/44100] ...
  [3600/44100] ...
  [3800/44100] ...
  [4000/44100] ...
  [4200/44100] ...
  [4400/44100] ...
  [4600/44100] ...
  [4800/44100] ...
  [5000/44100] ...
  [5200/44100] ...
  [5400/44100] ...
  [5600/44100] ...
  [5800/44100] ...
  [6000/44100] ...
  [6200/44100] ...
  [6400/44100] ...
  [6600/44100] ...
  [6800/44100] ...
  [7000/44100] ...
  [7200/44100] ...
  [7400/44100] ...
  [7600/44100] ...
  [7800/44100] ...
  [8000/44100] ...
  [8200/44100] ...
  [8400/44100] ...
  [8600/44100] ...
  [8800/44100] ...
  [9000/44100] ...
  [9200/44100] ...
  [9400/44100] ...
  [9600/44100] ...
  [9800/44100] ...
  [10000/44100] ...
  [10200/44100] ...
  [10400/44100] ...
  [10600/44100] ...
  [10800/44100] ...
  [11000/44100] ...
  [11200/44100] ...
  [11400/44100] ...
  [11600/44100] ...
  [

In [10]:
# Refined grid around switch-stage coarse best
fine_sw_tp = np.linspace(
    max(tau_prior_min, best_sw['tau_prior'] - tp_step),
    min(tau_prior_max, best_sw['tau_prior'] + tp_step),
    8,
)
fine_sw_eps = np.geomspace(
    max(eps_min, best_sw['epsilon'] / eps_ratio),
    min(eps_max, best_sw['epsilon'] * eps_ratio),
    8,
)
fine_sw_esw = refine_range(best_sw['epsilon_switch'], eps_switch_vals)
fine_sw_delta = refine_range(best_sw['delta'], delta_vals)
fine_sw_w = refine_range(best_sw['w'], w_vals)

total_sw_fine = len(fine_sw_tp) * len(fine_sw_eps) * len(fine_sw_esw) * len(fine_sw_delta) * len(fine_sw_w)
print(f'Refined grid (switch-stage): {total_sw_fine} points')
print(f'  tau_prior:  [{fine_sw_tp[0]:.4f}, {fine_sw_tp[-1]:.4f}]')
print(f'  epsilon:    [{fine_sw_eps[0]:.6f}, {fine_sw_eps[-1]:.6f}]')
print(f'  eps_switch: [{fine_sw_esw[0]:.4f}, {fine_sw_esw[-1]:.4f}]')
print(f'  delta:      [{fine_sw_delta[0]:.4f}, {fine_sw_delta[-1]:.4f}]')
print(f'  w:          [{fine_sw_w[0]:.4f}, {fine_sw_w[-1]:.4f}]')
print()

sw_refined = grid_search_mixture_switch(records, fine_sw_tp, fine_sw_eps,
                                         fine_sw_esw, fine_sw_delta, fine_sw_w)
sw_sweep.extend(sw_refined)
best_sw = pick_best(sw_sweep, metric)
print(f'\nRefined best: tau_prior={best_sw["tau_prior"]:.4f} eps={best_sw["epsilon"]:.6f} '
      f'eps_switch={best_sw["epsilon_switch"]:.4f} delta={best_sw["delta"]:.4f} w={best_sw["w"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

Refined grid (switch-stage): 13824 points
  tau_prior:  [2.5322, 6.5122]
  epsilon:    [0.117741, 0.200000]
  eps_switch: [0.8000, 1.0000]
  delta:      [0.0000, 0.0100]
  w:          [0.8000, 1.0000]

  [200/13824] ...
  [400/13824] ...
  [600/13824] ...
  [800/13824] ...
  [1000/13824] ...
  [1200/13824] ...
  [1400/13824] ...
  [1600/13824] ...
  [1800/13824] ...
  [2000/13824] ...
  [2200/13824] ...
  [2400/13824] ...
  [2600/13824] ...
  [2800/13824] ...
  [3000/13824] ...
  [3200/13824] ...
  [3400/13824] ...
  [3600/13824] ...
  [3800/13824] ...
  [4000/13824] ...
  [4200/13824] ...
  [4400/13824] ...
  [4600/13824] ...
  [4800/13824] ...
  [5000/13824] ...
  [5200/13824] ...
  [5400/13824] ...
  [5600/13824] ...
  [5800/13824] ...
  [6000/13824] ...
  [6200/13824] ...
  [6400/13824] ...
  [6600/13824] ...
  [6800/13824] ...
  [7000/13824] ...
  [7200/13824] ...
  [7400/13824] ...
  [7600/13824] ...
  [7800/13824] ...
  [8000/13824] ...
  [8200/13824] ...
  [8400/13824] ...
  [8

In [11]:
# Scipy polishing for switch-stage tuning
def objective_sw(params):
    tp, eps, eps_sw, delta, w = params
    res = evaluate_mixture_switch(records, tp, eps, eps_sw, delta, w)
    return -res[metric]

sw_tp_lo = max(tau_prior_min, best_sw['tau_prior'] - tp_step / 2)
sw_tp_hi = min(tau_prior_max, best_sw['tau_prior'] + tp_step / 2)
sw_eps_lo = max(1e-6, best_sw['epsilon'] / 2)
sw_eps_hi = min(eps_max, best_sw['epsilon'] * 2)
sw_esw_lo = max(0.01, best_sw['epsilon_switch'] - 0.1)
sw_esw_hi = min(1.0, best_sw['epsilon_switch'] + 0.1)
sw_delta_lo = max(0.0, best_sw['delta'] - 0.05)
sw_delta_hi = min(1.0, best_sw['delta'] + 0.05)
sw_w_lo = max(0.0, best_sw['w'] - 0.1)
sw_w_hi = min(1.0, best_sw['w'] + 0.1)

sw_x0 = [best_sw['tau_prior'], best_sw['epsilon'], best_sw['epsilon_switch'],
          best_sw['delta'], best_sw['w']]
sw_bounds = [(sw_tp_lo, sw_tp_hi), (sw_eps_lo, sw_eps_hi), (sw_esw_lo, sw_esw_hi),
             (sw_delta_lo, sw_delta_hi), (sw_w_lo, sw_w_hi)]

print('Scipy optimization (switch-stage) ...')
print(f'  bounds: tau_prior=[{sw_tp_lo:.4f}, {sw_tp_hi:.4f}], epsilon=[{sw_eps_lo:.6f}, {sw_eps_hi:.6f}]')
print(f'          eps_switch=[{sw_esw_lo:.4f}, {sw_esw_hi:.4f}], delta=[{sw_delta_lo:.4f}, {sw_delta_hi:.4f}], w=[{sw_w_lo:.4f}, {sw_w_hi:.4f}]')

sw_opt = minimize(objective_sw, sw_x0, method='L-BFGS-B', bounds=sw_bounds,
                  options={'maxiter': 50, 'ftol': 1e-6})
sw_opt_tp, sw_opt_eps, sw_opt_esw, sw_opt_delta, sw_opt_w = sw_opt.x
print(f'  Optimal: tau_prior={sw_opt_tp:.4f} eps={sw_opt_eps:.6f} eps_switch={sw_opt_esw:.4f} '
      f'delta={sw_opt_delta:.4f} w={sw_opt_w:.4f}')
print(f'  objective={-sw_opt.fun:.4f}')

sw_opt_eval = evaluate_mixture_switch(records, sw_opt_tp, sw_opt_eps, sw_opt_esw, sw_opt_delta, sw_opt_w)
sw_sweep.append(sw_opt_eval)
best_sw = pick_best(sw_sweep, metric)
print(f'\nFinal best (switch-stage): tau_prior={best_sw["tau_prior"]:.4f} eps={best_sw["epsilon"]:.6f} '
      f'eps_switch={best_sw["epsilon_switch"]:.4f} delta={best_sw["delta"]:.4f} w={best_sw["w"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

Scipy optimization (switch-stage) ...
  bounds: tau_prior=[3.2429, 5.2329], epsilon=[0.100000, 0.200000]
          eps_switch=[0.8600, 1.0000], delta=[0.0000, 0.0500], w=[0.9000, 1.0000]
  Optimal: tau_prior=4.2383 eps=0.200000 eps_switch=0.9794 delta=0.0000 w=1.0000
  objective=0.3910

Final best (switch-stage): tau_prior=4.2383 eps=0.200000 eps_switch=0.9794 delta=0.0000 w=1.0000
  combo_r=0.3910  marg_r=0.4890  mean_ll=-5.2063


## Save Parameters

In [12]:
output = {
    'metric_optimized': metric,
    'aggregate_tuned': {
        'tau_prior': best['tau_prior'], 'epsilon': best['epsilon'],
        'epsilon_switch': best['epsilon_switch'], 'delta': best['delta'], 'w': best['w'],
        'combo_r': best['combo_r'], 'marg_r': best['marg_r'], 'mean_ll': best['mean_ll'],
    },
    'switch_stage_tuned': {
        'tau_prior': best_sw['tau_prior'], 'epsilon': best_sw['epsilon'],
        'epsilon_switch': best_sw['epsilon_switch'], 'delta': best_sw['delta'], 'w': best_sw['w'],
        'combo_r': best_sw['combo_r'], 'marg_r': best_sw['marg_r'], 'mean_ll': best_sw['mean_ll'],
    },
}

with open('mixture_ps_params.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to mixture_ps_params.json')
print(json.dumps(output, indent=2))

Saved to mixture_ps_params.json
{
  "metric_optimized": "combo_r",
  "aggregate_tuned": {
    "tau_prior": 1.458253968253968,
    "epsilon": 0.004660091320263979,
    "epsilon_switch": 0.05,
    "delta": 0.0,
    "w": 0.68,
    "combo_r": 0.5238621958370322,
    "marg_r": 0.5295051008433598,
    "mean_ll": -9.332701643139238
  },
  "switch_stage_tuned": {
    "tau_prior": 4.238330416729619,
    "epsilon": 0.2,
    "epsilon_switch": 0.979362585048809,
    "delta": 0.0,
    "w": 1.0,
    "combo_r": 0.39100282573129447,
    "marg_r": 0.4890466318972992,
    "mean_ll": -5.206314254879205
  }
}
